# DNA Methylation Analysis with czip
Assuming we have single cell DNA methylation data with thousands of cells. In each cell, we have one allc file:

```text
chr1    3001733 +       CTG     0       1       1
chr1    3001743 +       CAA     0       1       1
chr1    3001746 +       CCT     0       1       1
chr1    3001747 +       CTG     0       1       1
chr1    3001752 +       CTG     0       1       1
chr1    3001758 +       CTT     0       1       1
chr1    3001761 +       CTG     0       1       1
chr1    3001764 +       CTT     0       1       1
chr1    3001768 +       CTT     0       1       1
chr1    3001776 +       CTG     0       1       1
```

Columns are chrom, position, strand, context, mc (methylated count), cov (coverage) and fraction.
The first four columns are shared among all cells, in this case, we don't need to store this redundant information for each cell, we can separate the reference and cell data and only store the mc and cov for each cell.
However, we can still query and view allc with reference information.

## Generate allc reference coordinate .cz file

```shell
time czip build_ref -g ~/Ref/mm10/mm10_ucsc_with_chrL.fa -O ~/Ref/mm10/annotations/mm10_with_chrL.allc.cz -n 20
# 0m48.859s
```

## View reference .cz file

In [1]:
!which czip

~/Software/conda/m3c/bin/czip


In [2]:
# view header
!czip header -I ~/Ref/mm10/annotations/mm10_with_chrL.allc.cz

magic  :  b'CZIP'
version  :  0.0
total_size  :  3074591415
message  :  /home/x-wding2/Ref/mm10/mm10_ucsc_with_chrL.fa
formats  :  ['Q', 'c', '3s']
columns  :  ['pos', 'strand', 'context']
dimensions  :  ['chrom']
header_size  :  98


In [3]:
# summary chunks
!czip summary -I ~/Ref/mm10/annotations/mm10_with_chrL.allc.cz

chrom	chunk_start_offset	chunk_size	chunk_tail_offset	chunk_nblocks	chunk_nrows
chr1	98	219469648	219585439	14459	78962721
chr10	219585439	146325915	365988448	9634	52609184
chr11	365988448	144624498	510689184	9527	52027265
chr12	510689184	135669225	646429919	8936	48799752
chr13	646429919	135593184	782094541	8927	48750883
chr14	782094541	138973134	921140929	9154	49987736
chr15	921140929	117415601	1038618416	7733	42230765
chr16	1038618416	108168134	1146843556	7123	38899643
chr17	1146843556	108852498	1255753436	7170	39153472
chr18	1255753436	100769279	1356575825	6636	36239315
chr19	1356575825	69185307	1425797586	4554	24870223
chr1_GL456210_random	1425797586	207711	1426005446	14	74695
chr1_GL456211_random	1426005446	295188	1426300831	20	106168
chr1_GL456212_random	1426300831	186768	1426487740	13	67179
chr1_GL456213_random	1426487740	48464	1426536273	4	17318
chr1_GL456221_random	1426536273	250282	1426786728	17	90146
chr2	1426786728	208653348	1635550033	13742	75046570
chr3	1635550033	1759319

Every chunk has a dimension (chrom, sample, cell types or the combination of those dimensions)

In [4]:
# view
! czip view -I ~/Ref/mm10/annotations/mm10_with_chrL.allc.cz --show-dim 0 | head

chrom	pos	strand	context
chr1	3000003	+	CTG
chr1	3000005	-	CAG
chr1	3000009	+	CTA
chr1	3000016	-	CAA
chr1	3000018	-	CAC
chr1	3000019	-	CCA
chr1	3000023	+	CTT
chr1	3000027	-	CAA
chr1	3000029	-	CTC


## Download example dataset

```shell
pip install pyfigshare
figshare download 25374073 -o czip_example_data
```

## Converting .allc.tsv.gz into .cz file

In [5]:
!czip allc2cz --help

usage: czip allc2cz [-h] -I INPUT -O OUTFILE [-r REFERENCE]
                    [--missing-value MISSING_VALUE] [-F FORMATS] [-C COLUMNS]
                    [-D DIMENSIONS] [-u USECOLS] [--pr PR] [--pa PA] [-s SEP]
                    [--path-to-chrom PATH_TO_CHROM] [-c CHUNKSIZE]

options:
  -h, --help            show this help message and exit
  -I INPUT, --input INPUT
                        input allc.tsv.gz (default: None)
  -O OUTFILE, --outfile OUTFILE
                        output .cz file (default: None)
  -r REFERENCE, --reference REFERENCE
                        reference .cz file (default: None)
  --missing-value MISSING_VALUE
                        missing value fill (default: [0, 0])
  -F FORMATS, --formats FORMATS
                        column formats (default: ['B', 'B'])
  -C COLUMNS, --columns COLUMNS
                        column names (default: ['mc', 'cov'])
  -D DIMENSIONS, --dimensions DIMENSIONS
                        dimension names (default: ['chrom'])


### Pack .allc.tsv.gz to .cz with coordinates (position)

In [6]:
! zcat ../czip_example_data/FC_E17a_3C_1-1-I3-F13.allc.tsv.gz | head

chr1	3001733	+	CTG	0	1	1
chr1	3001743	+	CAA	0	1	1
chr1	3001746	+	CCT	0	1	1
chr1	3001747	+	CTG	0	1	1
chr1	3001752	+	CTG	0	1	1
chr1	3001758	+	CTT	0	1	1
chr1	3001761	+	CTG	0	1	1
chr1	3001764	+	CTT	0	1	1
chr1	3001768	+	CTT	0	1	1
chr1	3001776	+	CTG	0	1	1

gzip: stdout: Broken pipe


```shell
time czip allc2cz -I ../czip_example_data/FC_E17a_3C_1-1-I3-F13.allc.tsv.gz -O ../czip_example_data/FC_E17a_3C_1-1-I3-F13.with_coordinate.cz -F Q,H,H -C pos,mc,cov -u 1,4,5
# 0m41.417s
```

In [7]:
! ls ../czip_example_data/FC_E17a_3C_1-1-I3-F13.allc.tsv.gz ../czip_example_data/FC_E17a_3C_1-1-I3-F13.with_coordinate.cz -sh

141M ../czip_example_data/FC_E17a_3C_1-1-I3-F13.allc.tsv.gz
 73M ../czip_example_data/FC_E17a_3C_1-1-I3-F13.with_coordinate.cz


In [8]:
!czip view -I ../czip_example_data/FC_E17a_3C_1-1-I3-F13.with_coordinate.cz --show-dim 0 |head

chrom	pos	mc	cov
chr1	3001733	0	1
chr1	3001743	0	1
chr1	3001746	0	1
chr1	3001747	0	1
chr1	3001752	0	1
chr1	3001758	0	1
chr1	3001761	0	1
chr1	3001764	0	1
chr1	3001768	0	1


### Pack .allc.tsv.gz to .cz without coordinates (using reference)

```shell
time czip allc2cz -I ../czip_example_data/FC_E17a_3C_1-1-I3-F13.allc.tsv.gz -O ../czip_example_data/FC_E17a_3C_1-1-I3-F13.cz -r ~/Ref/mm10/annotations/mm10_with_chrL.allc.cz
# 1m31.182s
```

In [9]:
! ls ../czip_example_data/FC_E17a_3C_1-1-I3-F13.allc.tsv.gz ../czip_example_data/FC_E17a_3C_1-1-I3-F13.cz -sh

141M ../czip_example_data/FC_E17a_3C_1-1-I3-F13.allc.tsv.gz
 30M ../czip_example_data/FC_E17a_3C_1-1-I3-F13.cz


In [10]:
# view .cz alone (no reference)
! czip view -I ../czip_example_data/FC_E17a_3C_1-1-I3-F13.cz --show-dim 0 |head

chrom	mc	cov
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0


In [11]:
# view FC_E17a_3C_1-1-I3-F13.cz together with reference
! czip view -I ../czip_example_data/FC_E17a_3C_1-1-I3-F13.cz --show-dim 0 -r ~/Ref/mm10/annotations/mm10_with_chrL.allc.cz |head

chrom	pos	strand	context	mc	cov
chr1	3000003	+	CTG	0	0
chr1	3000005	-	CAG	0	0
chr1	3000009	+	CTA	0	0
chr1	3000016	-	CAA	0	0
chr1	3000018	-	CAC	0	0
chr1	3000019	-	CCA	0	0
chr1	3000023	+	CTT	0	0
chr1	3000027	-	CAA	0	0
chr1	3000029	-	CTC	0	0


## Query

#### query allc.tsv.gz using tabix

In [12]:
!tabix ../czip_example_data/FC_E17a_3C_1-1-I3-F13.allc.tsv.gz chr9 | awk '$5 > 50' |head

chr9	3000294	-	CAT	54	63	1
chr9	3000342	-	CGA	69	85	1
chr9	3000354	-	CGT	77	82	1
chr9	3000381	+	CGT	52	64	1
chr9	3000382	-	CGG	74	87	1
chr9	3000399	+	CGA	66	67	1
chr9	3000441	+	CGT	84	138	1
chr9	3000457	+	CGT	139	162	1
chr9	3000458	-	CGA	64	74	1
chr9	3000472	+	CGT	161	183	1


#### query allc.cz using czip

In [13]:
# query cz (with coordinate)
! czip query -I ../czip_example_data/FC_E17a_3C_1-1-I3-F13.with_coordinate.cz -D chr9 -s 3000294 -e 3000472

chrom	pos	mc	cov
chr9	3000294	54	63
chr9	3000296	0	30
chr9	3000297	1	31
chr9	3000300	1	29
chr9	3000304	0	71
chr9	3000305	1	72
chr9	3000307	1	29
chr9	3000312	1	30
chr9	3000321	0	33
chr9	3000322	0	29
chr9	3000325	2	31
chr9	3000331	0	35
chr9	3000333	1	91
chr9	3000338	0	34
chr9	3000339	0	34
chr9	3000341	32	36
chr9	3000342	69	85
chr9	3000343	1	37
chr9	3000344	1	38
chr9	3000351	0	44
chr9	3000353	36	45
chr9	3000354	77	82
chr9	3000356	1	46
chr9	3000357	0	45
chr9	3000362	2	89
chr9	3000364	0	93
chr9	3000366	0	93
chr9	3000372	0	61
chr9	3000374	1	60
chr9	3000380	0	63
chr9	3000381	52	64
chr9	3000382	74	87
chr9	3000384	2	87
chr9	3000390	0	60
chr9	3000392	0	79
chr9	3000397	0	64
chr9	3000399	66	67
chr9	3000400	47	57
chr9	3000402	0	68
chr9	3000408	0	78
chr9	3000409	0	81
chr9	3000411	0	51
chr9	3000412	0	51
chr9	3000414	1	83
chr9	3000415	0	83
chr9	3000418	1	94
chr9	3000420	1	47
chr9	3000422	0	49
chr9	3000424	0	42
chr9	3000425	2	99
chr9	3000430	0	106
chr9	3000432	2	118
chr9	3000439	2	128
chr9	3000441	84	1

In [14]:
# query .cz file (without coordinate) together with reference
! czip query -I ../czip_example_data/FC_E17a_3C_1-1-I3-F13.cz -r ~/Ref/mm10/annotations/mm10_with_chrL.allc.cz -D chr9 -s 3000294 -e 3000472

chrom	pos	strand	context	mc	cov
chr9	3000294	-	CAT	54	63
chr9	3000296	+	CCT	0	30
chr9	3000297	+	CTA	1	31
chr9	3000300	+	CAA	1	29
chr9	3000304	-	CAT	0	71
chr9	3000305	-	CCA	1	72
chr9	3000307	+	CAT	1	29
chr9	3000312	+	CTA	1	30
chr9	3000321	+	CCA	0	33
chr9	3000322	+	CAA	0	29
chr9	3000325	+	CTT	2	31
chr9	3000331	+	CAG	0	35
chr9	3000333	-	CTG	1	91
chr9	3000338	+	CCT	0	34
chr9	3000339	+	CTC	0	34
chr9	3000341	+	CGC	32	36
chr9	3000342	-	CGA	69	85
chr9	3000343	+	CCA	1	37
chr9	3000344	+	CAT	1	38
chr9	3000351	+	CAC	0	44
chr9	3000353	+	CGT	36	45
chr9	3000354	-	CGT	77	82
chr9	3000356	+	CCT	1	46
chr9	3000357	+	CTA	0	45
chr9	3000362	-	CTT	2	89
chr9	3000364	-	CAC	0	93
chr9	3000366	-	CAC	0	93
chr9	3000372	+	CTC	0	61
chr9	3000374	+	CAT	1	60
chr9	3000380	+	CCG	0	63
chr9	3000381	+	CGT	52	64
chr9	3000382	-	CGG	74	87
chr9	3000384	-	CAC	2	87
chr9	3000390	+	CAG	0	60
chr9	3000392	-	CTG	0	79
chr9	3000397	+	CTC	0	64
chr9	3000399	+	CGA	66	67
chr9	3000400	-	CGA	47	57
chr9	3000402	+	CAT	0	68
chr9	3000408	+	CCA	0	78

## Cat multiple .cz files into one .cz file

In [15]:
!czip catcz -h

usage: czip catcz [-h] -O OUTPUT -I INPUT [-F FORMATS] [-C COLUMNS]
                  [-D DIMENSIONS] [--dim-order DIM_ORDER] [--add-dim]
                  [--title TITLE] [-m MESSAGE]

options:
  -h, --help            show this help message and exit
  -O OUTPUT, --output OUTPUT
                        output .cz file (default: None)
  -I INPUT, --input INPUT
                        input pattern or comma-separated .cz paths (default:
                        None)
  -F FORMATS, --formats FORMATS
                        column formats (default: ['B', 'B'])
  -C COLUMNS, --columns COLUMNS
                        column names (default: ['mc', 'cov'])
  -D DIMENSIONS, --dimensions DIMENSIONS
                        dimension names (default: ['chrom'])
  --dim-order DIM_ORDER
                        dimension order file or comma-separated (default:
                        None)
  --add-dim             add filename as extra dimension (default: False)
  --title TITLE         title for added d

```shell
czip catcz -O all_cell_type.cz -F Q,c,3s -C pos,strand,context -D chrom -I "cell_type/*.cz" \
            --dim_order ~/Ref/mm10/mm10_ucsc_with_chrL.chrom.sizes --add_dim True --title "cell_id"
```

In this example, we cat multiple .cz file into one .cz file and add another dimension to each chunk (cell_id)

## Extract CG from .cz and merge strand

In [16]:
! czip extractCG --help

usage: czip extractCG [-h] -I INPUT -O OUTFILE -s SSI [-c CHUNKSIZE]
                      [--merge-cg]

options:
  -h, --help            show this help message and exit
  -I INPUT, --input INPUT
                        input .cz file (default: None)
  -O OUTFILE, --outfile OUTFILE
                        output .cz file (default: None)
  -s SSI, --ssi SSI     CGN subset index file (default: None)
  -c CHUNKSIZE, --chunksize CHUNKSIZE
                        rows per chunk (default: 5000)
  --merge-cg            merge forward/reverse CG (default: False)


```shell
czip extractCG -I cz/FC_P13a_3C_2-1-E5-D13.cz -O FC_P13a_3C_2-1-E5-D13.CGN.cz -s ~/Ref/mm10/annotations/mm10_with_chrL.allc.cz.CGN.ssi

# view CG .cz
czip view -I FC_P13a_3C_2-1-E5-D13.CGN.cz --show-dim 0 -r ~/Ref/mm10/annotations/mm10_with_chrL.allCG.forward.cz
```

## Merge multiple .cz files into one .cz file
sum up the mc and cov

In [17]:
!czip merge_cz --help

usage: czip merge_cz [-h] [-i INDIR] [--cz-paths CZ_PATHS]
                     [--class-table CLASS_TABLE] [-O OUTFILE]
                     [--prefix PREFIX] [-n N_JOBS] [-F FORMATS]
                     [--path-to-chrom PATH_TO_CHROM] [-r REFERENCE]
                     [--keep-cat] [--batchsize BATCHSIZE] [--temp]
                     [--no-bgzip] [-c CHUNKSIZE] [--ext EXT]

options:
  -h, --help            show this help message and exit
  -i INDIR, --indir INDIR
                        input directory (default: None)
  --cz-paths CZ_PATHS   file listing .cz paths (default: None)
  --class-table CLASS_TABLE
                        cell class table (default: None)
  -O OUTFILE, --outfile OUTFILE
                        output file (default: None)
  --prefix PREFIX       output prefix (default: None)
  -n N_JOBS, --n-jobs N_JOBS
                        parallel jobs (default: 12)
  -F FORMATS, --formats FORMATS
                        output formats (default: ['H', 'H'])
  --path-to

```shell
czip merge_mz -i cz-CGN/ -O merged.cz
```

## Merge .cz files belonging to the same cell type
sum up the mc and cov

In [18]:
!czip merge_cell_type --help

usage: czip merge_cell_type [-h] [-i INDIR] [--cell-table CELL_TABLE]
                            [-O OUTDIR] [-n N_JOBS]
                            [--path-to-chrom PATH_TO_CHROM] [--ext EXT]

options:
  -h, --help            show this help message and exit
  -i INDIR, --indir INDIR
                        input directory (default: None)
  --cell-table CELL_TABLE
                        cell-type table (default: None)
  -O OUTDIR, --outdir OUTDIR
                        output directory (default: None)
  -n N_JOBS, --n-jobs N_JOBS
                        parallel jobs (default: 64)
  --path-to-chrom PATH_TO_CHROM
                        chrom order file (default: None)
  --ext EXT             input file extension (default: .CGN.merged.cz)


## Converting .cz file back to allc.tsv.gz

convert both CG and CH to allc

```shell
czip view -I ../czip_example_data/FC_E17a_3C_1-1-I3-F13.cz --show-dim 0 -r ~/Ref/mm10/annotations/mm10_with_chrL.allc.cz --no-header | 
    awk 'BEGIN{FS=OFS="\t"}; {print $0,1}' | bgzip > test1.allc.tsv.gz && tabix -f -s 1 -b 2 -e 2 test1.allc.tsv.gz
```

convert only CG to allc

```shell
czip view -I ../czip_example_data/FC_E17a_3C_1-1-I3-F13.cz --show-dim 0 -r ~/Ref/mm10/annotations/mm10_with_chrL.allCG.forward.cz --no-header | 
    awk 'BEGIN{FS=OFS=\"\t\"}; {print \$0,1}' | bgzip > test1.CG.allc.tsv.gz && tabix -f -s 1 -b 2 -e 2 test1.CG.allc.tsv.gz
```